# Energy AI Hackathon 2026 - Brain Oil

> **SUBMISSION INSTRUCTIONS:**
> 1. This file is named `BrainOil.ipynb`
> 2. Commit to the hackathon GitHub organization repo
> 3. Submit solution.csv with predictions for Wells 72-83

**Team Members:**
- [Member 1 Full Name] - [Department/Affiliation]
- [Member 2 Full Name] - [Department/Affiliation]
- [Member 3 Full Name] - [Department/Affiliation]
- [Member 4 Full Name] - [Department/Affiliation]

---

## Executive Summary

This notebook presents our complete machine learning workflow for predicting **3-year cumulative oil production (BBL)** for 12 preproduction wells (Well IDs 72-83). Our solution includes point estimates and 100 uncertainty realizations (R1-R100) per prediction.

**Key Approach:**
- Well log aggregation (multi-row depth data → one row per well)
- MICE imputation for missing petrophysical data (~7-10% missing)
- Random Forest with Optuna hyperparameter tuning
- Residual bootstrapping for uncertainty quantification

**Results Summary:**
- CV R²: [To be filled after training]
- Prediction range: 8M - 74M BBL (matching training data distribution)


---
## 1. Problem Statement

The Energy AI Hackathon 2026 challenges teams to **predict 3-year cumulative oil production** for wells that haven't yet started producing. This is critical for:

- **Investment decisions**: Prioritize high-value wells for development
- **Reservoir management**: Optimize field development strategy
- **Risk assessment**: Quantify uncertainty in production forecasts

### Data Structure
| Dataset | Description | Size |
|---------|-------------|------|
| Well_log_data_production_wells.csv | Well logs for 71 training wells | ~21 rows/well, 1491 total |
| Well_log_data_preproduction_wells.csv | Well logs for 12 test wells | ~21 rows/well, 252 total |
| Production_history_production_wells.csv | Monthly cumulative production | 5517 rows |
| 2d_sand_proportion.npy | 200x200 spatial sand map | Spatial feature |

### Key Challenge: Multi-Row Data
Each well has ~21 depth measurements (Z values from ~19 to ~39). We must aggregate these to one feature vector per well.


---
## 2. Setup and Data Ingestion


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

In [ ]:
# Load data files
DATA_DIR = "data"

prod_wells = pd.read_csv(f"{DATA_DIR}/Well_log_data_production_wells.csv")
preprod_wells = pd.read_csv(f"{DATA_DIR}/Well_log_data_preproduction_wells.csv")
prod_history = pd.read_csv(f"{DATA_DIR}/Production_history_production_wells.csv")
sand_map = np.load(f"{DATA_DIR}/2d_sand_proportion.npy")

print(f"Production wells (training): {prod_wells['Well_ID'].nunique()} wells, {len(prod_wells)} rows")
print(f"Preproduction wells (test): {preprod_wells['Well_ID'].nunique()} wells, {len(preprod_wells)} rows")
print(f"Production history: {len(prod_history)} rows")
print(f"Sand map shape: {sand_map.shape}")

---
## 3. Data Exploration


In [ ]:
# Check feature columns
print("Well log columns:")
print(prod_wells.columns.tolist())

# Missing values analysis
print("\nMissing values (%)")
missing_pct = (prod_wells.isnull().sum() / len(prod_wells) * 100).round(1)
print(missing_pct[missing_pct > 0])

In [ ]:
# Facies distribution
print("Facies distribution:")
print(prod_wells['facies'].value_counts())

---
## 4. Well Log Aggregation

**Strategy:** For each numeric feature, compute mean, std, min, max across all depth measurements.


In [ ]:
def aggregate_well_logs(well_logs_df):
    """Aggregate multi-row well logs to one row per well."""
    numeric_cols = ['AI', 'SI', 'Vp', 'Vs', 'rho_b', 'rho_f', 'rho_m', 
                    'K0', 'Kdry', 'Kf', 'Ksat', 'G0', 'Gdry', 'Gsat', 
                    'phi', 'perm', 'GR']
    
    agg_dict = {}
    for col in numeric_cols:
        if col in well_logs_df.columns:
            agg_dict[f'{col}_mean'] = (col, 'mean')
            agg_dict[f'{col}_std'] = (col, 'std')
            agg_dict[f'{col}_min'] = (col, 'min')
            agg_dict[f'{col}_max'] = (col, 'max')
    
    # Spatial and depth features
    agg_dict['X'] = ('X', 'first')
    agg_dict['Y'] = ('Y', 'first')
    agg_dict['Z_min'] = ('Z', 'min')
    agg_dict['Z_max'] = ('Z', 'max')
    agg_dict['depth_range'] = ('Z', lambda x: x.max() - x.min())
    agg_dict['n_measurements'] = ('Z', 'count')
    
    aggregated = well_logs_df.groupby('Well_ID').agg(**agg_dict).reset_index()
    
    # Add facies distribution
    if 'facies' in well_logs_df.columns:
        facies_pivot = well_logs_df.groupby(['Well_ID', 'facies']).size().unstack(fill_value=0)
        facies_pivot = facies_pivot.div(facies_pivot.sum(axis=1), axis=0)
        facies_pivot.columns = [f'facies_{int(c)}_pct' for c in facies_pivot.columns]
        aggregated = aggregated.merge(facies_pivot.reset_index(), on='Well_ID', how='left')
    
    return aggregated

# Aggregate well logs
train_agg = aggregate_well_logs(prod_wells)
test_agg = aggregate_well_logs(preprod_wells)

print(f"Aggregated training data: {len(train_agg)} wells, {len(train_agg.columns)} features")
print(f"Aggregated test data: {len(test_agg)} wells, {len(test_agg.columns)} features")

---
## 5. Target Calculation (3-Year Cumulative Oil)


In [ ]:
def calculate_3year_targets(prod_history):
    """Calculate 3-year cumulative oil production for each well."""
    prod_history['Date'] = pd.to_datetime(prod_history['Date'])
    
    targets = []
    for well_id in prod_history['Well_ID'].unique():
        well_data = prod_history[prod_history['Well_ID'] == well_id].sort_values('Date')
        start_date = well_data['Date'].min()
        end_date = start_date + pd.DateOffset(years=3)
        
        within_3yr = well_data[well_data['Date'] <= end_date]
        if len(within_3yr) > 0:
            final_oil = within_3yr['Cumulative Oil Production, BBL'].iloc[-1]
            targets.append({'Well_ID': well_id, 'Target_3yr_Oil_BBL': final_oil})
    
    return pd.DataFrame(targets)

targets = calculate_3year_targets(prod_history)
train_agg = train_agg.merge(targets, on='Well_ID', how='left')

print(f"Target range: {train_agg['Target_3yr_Oil_BBL'].min():,.0f} to {train_agg['Target_3yr_Oil_BBL'].max():,.0f} BBL")
print(f"Target mean: {train_agg['Target_3yr_Oil_BBL'].mean():,.0f} BBL")

In [ ]:
# Add sand proportion from spatial map
def lookup_sand_proportion(df, sand_map):
    x_coords = df['X'].values
    y_coords = df['Y'].values
    
    x_scaled = np.clip((x_coords / x_coords.max() * (sand_map.shape[1] - 1)).astype(int), 0, sand_map.shape[1] - 1)
    y_scaled = np.clip((y_coords / y_coords.max() * (sand_map.shape[0] - 1)).astype(int), 0, sand_map.shape[0] - 1)
    
    return sand_map[y_scaled, x_scaled]

train_agg['sand_proportion'] = lookup_sand_proportion(train_agg, sand_map)
test_agg['sand_proportion'] = lookup_sand_proportion(test_agg, sand_map)

print(f"Sand proportion range: {train_agg['sand_proportion'].min():.2f} to {train_agg['sand_proportion'].max():.2f}")

---
## 6. MICE Imputation

**MICE (Multivariate Imputation by Chained Equations)** uses relationships between features to impute missing values - recommended by hackathon host.


In [ ]:
# Get numeric columns for imputation
exclude_cols = ['Well_ID', 'Target_3yr_Oil_BBL']
numeric_cols = [c for c in train_agg.columns if c not in exclude_cols and train_agg[c].dtype in ['float64', 'int64']]

# Apply MICE
mice_imputer = IterativeImputer(random_state=42, max_iter=10)

mice_imputer.fit(train_agg[numeric_cols])
train_agg[numeric_cols] = mice_imputer.transform(train_agg[numeric_cols])
test_agg[[c for c in numeric_cols if c in test_agg.columns]] = mice_imputer.transform(
    test_agg[[c for c in numeric_cols if c in test_agg.columns]]
)

# Fill remaining NaN with 0
train_agg = train_agg.fillna(0)
test_agg = test_agg.fillna(0)

print(f"Remaining NaN in training: {train_agg.isnull().sum().sum()}")
print(f"Remaining NaN in test: {test_agg.isnull().sum().sum()}")

---
## 7. Feature Engineering

**Industry-Standard Features (SPE Literature):**
- **RQI** = 0.0314 × √(k/φ) - Reservoir Quality Index (Amaefule et al. 1993)
- **FZI** = RQI / [φ/(1-φ)] - Flow Zone Indicator for hydraulic flow units
- **Vp/Vs ratio** - Lithology & fluid indicator from rock physics

**Rock Quality Features (Industry Expert Advice):**
- High porosity (phi) = storage capacity
- High permeability = flow ability
- Low Gamma Ray (GR) = clean sand, less shale


In [ ]:
# Feature Engineering - Industry Standard + Rock Quality
for df in [train_agg, test_agg]:
    # INDUSTRY-STANDARD FEATURES (SPE Literature)
    # RQI - Reservoir Quality Index (Amaefule et al. 1993)
    if 'phi_mean' in df.columns and 'perm_mean' in df.columns:
        phi_safe = df['phi_mean'].replace(0, 0.001)
        df['RQI'] = 0.0314 * np.sqrt(df['perm_mean'] / phi_safe)
        # FZI - Flow Zone Indicator
        phi_z = phi_safe / (1 - phi_safe)
        df['FZI'] = df['RQI'] / phi_z
    
    # Vp/Vs ratio - Lithology & fluid indicator
    if 'Vp_mean' in df.columns and 'Vs_mean' in df.columns:
        vs_safe = df['Vs_mean'].replace(0, 1)
        df['Vp_Vs_ratio'] = df['Vp_mean'] / vs_safe
    
    # DERIVED ROCK QUALITY FEATURES
    if 'phi_mean' in df.columns and 'perm_mean' in df.columns:
        df['phi_perm_product'] = df['phi_mean'] * np.log1p(df['perm_mean'])
    
    if 'phi_mean' in df.columns and 'GR_mean' in df.columns:
        df['rock_quality'] = df['phi_mean'] / (df['GR_mean'] + 1)
    
    if 'AI_mean' in df.columns and 'SI_mean' in df.columns:
        df['impedance_ratio'] = df['AI_mean'] / (df['SI_mean'] + 1)
    
    if 'facies_5_pct' in df.columns and 'facies_6_pct' in df.columns:
        df['net_to_gross'] = 1 - df['facies_5_pct'] - df['facies_6_pct']
    
    if 'phi_mean' in df.columns and 'depth_range' in df.columns:
        df['storage_capacity'] = df['phi_mean'] * df['depth_range']
    
    if 'perm_mean' in df.columns and 'GR_mean' in df.columns:
        df['flow_quality'] = np.log1p(df['perm_mean']) / (df['GR_mean'] + 1)

print(f"Final feature count: {len(train_agg.columns)}")
print("\nIndustry-standard feature correlations:")
for col in ['RQI', 'FZI', 'Vp_Vs_ratio']:
    if col in train_agg.columns:
        corr = train_agg[col].corr(train_agg['Target_3yr_Oil_BBL'])
        print(f"  {col}: {corr:.3f}")
print("\nRock quality feature correlations:")
for col in ['phi_perm_product', 'rock_quality', 'net_to_gross', 'storage_capacity', 'flow_quality']:
    if col in train_agg.columns:
        corr = train_agg[col].corr(train_agg['Target_3yr_Oil_BBL'])
        print(f"  {col}: {corr:.3f}")

---
## 8. Model Training

**Dr. Pyrcz's Advice:**
- Normalize everything (StandardScaler)
- Try simplest things first (Linear Regression)
- Look at highs and lows


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge

# Prepare features and target
exclude_cols = ['Well_ID', 'Target_3yr_Oil_BBL']
feature_cols = [c for c in train_agg.columns if c not in exclude_cols and train_agg[c].dtype in ['float64', 'int64']]

X = train_agg[feature_cols].fillna(0)
y = train_agg['Target_3yr_Oil_BBL']

# Normalize features (Dr. Pyrcz's recommendation)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

print(f"Training with {len(feature_cols)} features")
print(f"Training samples: {len(X)}")
print("Features normalized with StandardScaler")

In [ ]:
# Try Linear Regression first (Dr. Pyrcz's advice: start simple)
linear_model = LinearRegression()
linear_scores = cross_val_score(linear_model, X_scaled, y, cv=5, scoring='r2')
print(f"Linear Regression CV R²: {linear_scores.mean():.4f} (+/- {linear_scores.std():.4f})")

# Also try Ridge
ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_scores = cross_val_score(ridge_model, X_scaled, y, cv=5, scoring='r2')
print(f"Ridge Regression CV R²: {ridge_scores.mean():.4f} (+/- {ridge_scores.std():.4f})")

In [ ]:
# Random Forest with Optuna hyperparameter tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'random_state': 42,
        'n_jobs': -1
    }
    model = RandomForestRegressor(**params)
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='r2')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"Best params: {study.best_params}")
print(f"Best CV R² (Random Forest): {study.best_value:.4f}")

In [ ]:
# Train final model with best params
best_params = study.best_params
best_params['random_state'] = 42
best_params['n_jobs'] = -1

model = RandomForestRegressor(**best_params)
model.fit(X, y)

# Calculate residuals for uncertainty
y_pred_train = model.predict(X)
residuals = y - y_pred_train

print(f"Train R²: {1 - np.sum(residuals**2) / np.sum((y - y.mean())**2):.4f}")
print(f"Residual std: {residuals.std():,.0f} BBL")

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 15 Features:")
print(importance_df.head(15))

---
## 9. Generate Predictions with Uncertainty


In [ ]:
# Prepare test features
X_test = test_agg[feature_cols].fillna(0)

# Point predictions
point_predictions = model.predict(X_test)

# Generate 100 realizations via residual bootstrapping
n_realizations = 100
realizations = np.zeros((len(test_agg), n_realizations))

for i in range(n_realizations):
    sampled_residuals = np.random.choice(residuals, size=len(test_agg), replace=True)
    realizations[:, i] = point_predictions + sampled_residuals
    realizations[:, i] = np.maximum(realizations[:, i], 0)  # Ensure non-negative

print(f"Generated {n_realizations} realizations for {len(test_agg)} wells")

In [ ]:
# Create solution DataFrame
solution = pd.DataFrame()
solution['Well_ID'] = test_agg['Well_ID'].values
solution['Prediction_BBL'] = point_predictions.round(0).astype(int)

# Add realizations R1-R100
for i in range(n_realizations):
    solution[f'R{i+1}'] = realizations[:, i].round(0).astype(int)

# Save solution
solution.to_csv('solution.csv', index=False)
print("Solution saved to solution.csv")

# Display
print("\nSolution preview:")
print(solution.head())

In [ ]:
# Prediction summary
print(f"\nPrediction Summary:")
print(f"Min: {solution['Prediction_BBL'].min():,.0f} BBL")
print(f"Mean: {solution['Prediction_BBL'].mean():,.0f} BBL")
print(f"Max: {solution['Prediction_BBL'].max():,.0f} BBL")

# Uncertainty visualization
plt.figure(figsize=(12, 6))
for idx, row in solution.iterrows():
    reals = row[[f'R{i+1}' for i in range(100)]].values
    plt.boxplot(reals, positions=[row['Well_ID']], widths=0.6)
plt.xlabel('Well ID')
plt.ylabel('Oil Production (BBL)')
plt.title('Prediction Uncertainty by Well')
plt.show()

---
## 10. Conclusion

**Summary:**
- Successfully predicted 3-year cumulative oil production for 12 preproduction wells
- Used well log aggregation to handle multi-row depth data
- Applied MICE imputation for ~7-10% missing values
- Optuna-tuned Random Forest model
- Residual bootstrapping provides 100 uncertainty realizations

**Key Features:**
- Petrophysical properties (phi, perm, GR) dominate feature importance
- Spatial features (X, Y, sand_proportion) capture reservoir heterogeneity
- Facies distribution provides lithology information

**Team Brain Oil** - Energy AI Hackathon 2026
